# DIT5411 — Traditional Chinese Handwriting Recognition (Kaggle)

**Before running:** Settings → Accelerator → **GPU P100**, and Settings → Internet → **On**.

Kaggle gives 30 GPU-hours/week. The dataset is downloaded inside the notebook — no upload needed.
Full run = 3 models × 3 epochs at 128px ≈ 22 GPU-hours, fits in one week.

Checkpoints go to `/kaggle/working/results/checkpoints`. Click **Save Version** during/after training to keep them as Output; later open that version → **Copy & Run Code** to resume via `--resume`.

In [ ]:
import os, sys, subprocess, glob

if not os.path.isfile('train.py'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/cveg1234/DIT5411-LuiYuHin.git'], check=True)
    os.chdir('DIT5411-LuiYuHin')
subprocess.run(['git', 'pull', '-q'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('Working dir:', os.getcwd())
print('GPU devices:', gpus)
if not gpus:
    print('WARNING: no GPU detected. Set Settings -> Accelerator -> GPU P100 before the full run.')
print('Setup done.')

## Download & extract the dataset (one time, ~15 min)
684,677 images across 13,065 characters. If `data/cleaned_data` already exists (e.g. from a saved Output), this cell skips.

In [ ]:
DATA_DIR = os.path.join(os.getcwd(), 'data', 'cleaned_data')
if not os.path.isdir(DATA_DIR) or len(os.listdir(DATA_DIR)) < 1000:
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/chenkenanalytic/handwritting_data_all.git'], check=True)
    with open('all_data.zip', 'wb') as out:
        for part in sorted(glob.glob('handwritting_data_all/all_data.zip.*')):
            with open(part, 'rb') as f:
                out.write(f.read())
    subprocess.run([sys.executable, 'extract_dataset.py',
                    '--zip', 'all_data.zip', '--dest', DATA_DIR], check=True)
print('Character folders:', len(os.listdir(DATA_DIR)))

## Quick pilot (optional — 20 classes, 2 epochs)

In [ ]:
r = subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                    '--output-dir', 'results_pilot', '--num-classes', '20',
                    '--epochs', '2', '--models', 'A', 'B', '--no-imagenet',
                    '--steps-per-epoch', '30'])
print('pilot exit code:', r.returncode)
print(open('results_pilot/RESULTS.md', encoding='utf-8').read())

## Train models one at a time
Each cell trains **one model for 2 epochs** with `--resume`, then prints its result. Run A, then B, then C. If a session dies, Quick Save + Copy & Run and re-run the remaining cells — finished models are kept in `results.json` and their checkpoints are reused.

Each cell takes ~5 GPU-hours (2 epochs × 20,394 steps).

In [ ]:
# ---- Model A (2 epochs, ~5h) ----
r = subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                    '--output-dir', 'results', '--epochs', '2',
                    '--models', 'A', '--resume'])
print('Model A exit code:', r.returncode)
if os.path.isfile('results/RESULTS.md'):
    print(open('results/RESULTS.md', encoding='utf-8').read())
else:
    print('Not finished — check checkpoints:')
    import glob as _g
    for f in sorted(_g.glob('results/checkpoints/*_state.json')):
        print(' ', os.path.basename(f))

In [ ]:
# ---- Model B (2 epochs, ~5h) ----
r = subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                    '--output-dir', 'results', '--epochs', '2',
                    '--models', 'B', '--resume'])
print('Model B exit code:', r.returncode)
if os.path.isfile('results/RESULTS.md'):
    print(open('results/RESULTS.md', encoding='utf-8').read())
else:
    print('Not finished — check checkpoints:')
    import glob as _g
    for f in sorted(_g.glob('results/checkpoints/*_state.json')):
        print(' ', os.path.basename(f))

In [ ]:
# ---- Model C (2 epochs, ~5h) ----
r = subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                    '--output-dir', 'results', '--epochs', '2',
                    '--models', 'C', '--resume'])
print('Model C exit code:', r.returncode)
if os.path.isfile('results/RESULTS.md'):
    print(open('results/RESULTS.md', encoding='utf-8').read())
else:
    print('Not finished — check checkpoints:')
    import glob as _g
    for f in sorted(_g.glob('results/checkpoints/*_state.json')):
        print(' ', os.path.basename(f))

## Final results (after A, B and C are done)

In [ ]:
from IPython.display import Markdown, display
p = 'results/RESULTS.md'
display(Markdown(open(p, encoding='utf-8').read()) if os.path.isfile(p)
        else 'RESULTS.md not generated yet — finish all models first.')

## Publish to GitHub (automatic)
Add a GitHub Personal Access Token (scope: **repo**) as a Kaggle Secret named **`GH_TOKEN`**: Notebook settings → Secrets → Add secret. Then run the cell below — it copies `RESULTS.md`, `results.json` and the label map into the repo, updates the README accuracy section, commits and pushes. The token is read from the environment and never stored in the notebook or repo.

In [ ]:
import subprocess as _sp
if not os.environ.get('GH_TOKEN'):
    print('GH_TOKEN secret missing — add it in Settings -> Secrets, then re-run.')
else:
    r = _sp.run([sys.executable, 'publish_results.py'])
    print('publish exit code:', r.returncode)